# Síntesis de voz: de texto a audio

**Lección 2 · Clase 5.3** — el camino inverso al de la lección 1. Ahí teníamos una onda y sacábamos texto; acá entra texto y sale una onda.

Podría parecer la mitad aburrida del par, y hasta hace poco lo era: se elegía una voz de un catálogo, se mandaba el texto y salía lo que salía. Lo que cambió —y es el centro de esta lección— es que hoy la síntesis se **dirige**. Al modelo se le da el texto *y una instrucción de actuación en lenguaje natural*: "habla como un ejecutivo tranquilizando a un cliente molesto", "susurra", "lee esto como un locutor de radio deportiva". El mismo texto, con la misma voz, sale distinto.

Eso convierte al TTS de un componente de infraestructura en una decisión de producto, porque el tono de una voz es lo que la gente percibe como la personalidad de tu empresa.

El plan:

| | |
|---|---|
| **Lo básico** | `gpt-4o-mini-tts`: una llamada, un archivo de audio. |
| **La dirección** | El parámetro `instructions`: el mismo texto, cuatro actuaciones. |
| **El catálogo** | Las 13 voces de OpenAI. |
| **El especialista** | ElevenLabs `eleven_v3`: catálogo enorme, acentos y clonación. |
| **La decisión** | Costo y latencia medidos, y cuándo conviene cada uno. |

In [ ]:
# Esta lección usa el entorno uv del README. Si la corres en Colab, descomenta:
# %pip install -q openai==2.52.0 elevenlabs==2.60.0 python-dotenv==1.2.2
from dotenv import load_dotenv
import os

# Carga las llaves desde .env si existe (local); en Colab usa Secrets.
load_dotenv()

try:
    from google.colab import userdata  # type: ignore
    for llave in ("OPENAI_API_KEY", "ELEVENLABS_API_KEY", "ELEVEN_API_KEY"):
        try:
            valor = userdata.get(llave)
            if valor:
                os.environ[llave] = valor
        except Exception:
            pass
except Exception:
    pass

# El material antiguo del curso usaba ELEVEN_API_KEY; el SDK actual espera ELEVENLABS_API_KEY.
if not os.environ.get("ELEVENLABS_API_KEY") and os.environ.get("ELEVEN_API_KEY"):
    os.environ["ELEVENLABS_API_KEY"] = os.environ["ELEVEN_API_KEY"]

HAY_OPENAI = bool(os.environ.get("OPENAI_API_KEY"))
HAY_ELEVEN = bool(os.environ.get("ELEVENLABS_API_KEY"))
print("OPENAI_API_KEY presente    :", HAY_OPENAI)
print("ELEVENLABS_API_KEY presente:", HAY_ELEVEN)
if not HAY_ELEVEN:
    print("⚠️ ElevenLabs es opcional: crea una cuenta gratis en elevenlabs.io (10.000 caracteres/mes)")
    print("   o simplemente salta esa sección — el resto del notebook funciona igual.")

from pathlib import Path

SALIDAS = Path("outputs")
SALIDAS.mkdir(exist_ok=True)

## Un poco de historia, para saber qué estamos usando

Vale la pena ubicar el modelo en su linaje, porque explica por qué recién ahora se puede dirigir la actuación:

1. **Síntesis concatenativa** (años 90-2000): se grababa a un actor pronunciando miles de fragmentos y el sistema los pegaba. Inteligible, pero con esas costuras que uno reconocía al instante en los teléfonos de los bancos. Cambiar la emoción implicaba **volver a grabar al actor**.
2. **Síntesis paramétrica**: un modelo estadístico generaba parámetros del habla y un vocoder los convertía en audio. Más flexible, más robótico.
3. **La era neuronal** (2016 en adelante): **WaveNet** genera la muestra de audio directamente, y **Tacotron** aprende de texto a espectrograma de punta a punta. Acá la calidad se acerca a la humana por primera vez.
4. **Hoy**: modelos que comparten arquitectura y entrenamiento con los LLMs. Y por eso entienden una instrucción escrita sobre *cómo* decir la frase — es la misma capacidad de seguir instrucciones que ya conocemos en el texto, aplicada a la prosodia.

El modelo que vamos a usar es **`gpt-4o-mini-tts`**. Nota que los modelos que probablemente hayas visto en tutoriales — `tts-1` y `tts-1-hd` — son la generación anterior: siguen funcionando pero tienen menos voces y **no aceptan `instructions`**.

In [ ]:
from openai import OpenAI
from IPython.display import Audio, display

MODELO_TTS = "gpt-4o-mini-tts"
cliente = OpenAI() if HAY_OPENAI else None

TEXTO = "Hola, y bienvenidos al Diplomado de Inteligencia Artificial Generativa para Organizaciones."


def hablar(texto: str, voz: str = "marin", instrucciones: str | None = None,
           nombre_archivo: str = "voz.mp3") -> Path | None:
    """Sintetiza `texto` con gpt-4o-mini-tts y devuelve la ruta del audio."""
    if not HAY_OPENAI:
        print("⚠️ Falta OPENAI_API_KEY — se salta la síntesis.")
        return None

    destino = SALIDAS / nombre_archivo
    extras = {"instructions": instrucciones} if instrucciones else {}

    # with_streaming_response evita cargar todo el audio en memoria.
    # (El viejo `response.stream_to_file(...)` está deprecado.)
    with cliente.audio.speech.with_streaming_response.create(
        model=MODELO_TTS,
        voice=voz,
        input=texto,
        response_format=destino.suffix.lstrip("."),  # mp3 o wav según la extensión
        **extras,
    ) as respuesta:
        respuesta.stream_to_file(destino)

    return destino


ruta = hablar(TEXTO, voz="marin", nombre_archivo="01_basico.mp3")
if ruta:
    print(f"{ruta} · {ruta.stat().st_size / 1024:.0f} KB")
    display(Audio(filename=str(ruta)))

## `instructions`: dirigir la actuación

Acá está lo nuevo. El parámetro `instructions` recibe una descripción en lenguaje natural de **cómo** debe sonar, y controla acento, rango emocional, entonación, imitaciones, velocidad, tono y volumen.

Para verlo con claridad usamos un texto sacado a propósito de un contexto real y **neutro** — la frase que diría un sistema de atención al cliente. Es el caso donde el tono importa de verdad: la misma información puede sonar a "estamos en esto contigo" o a "trámite número 4.782".

La misma frase, la misma voz, cuatro direcciones distintas.

In [ ]:
FRASE = (
    "Su solicitud quedó registrada con el folio cuatro, siete, dos. "
    "Le vamos a responder dentro de las próximas cuarenta y ocho horas."
)

DIRECCIONES = {
    "neutra": None,
    "ejecutivo_calido": (
        "Habla como un ejecutivo de atención al cliente chileno, cálido y tranquilizador, "
        "hablándole a alguien que llamó preocupado. Ritmo pausado, tono empático y cercano."
    ),
    "locutor_energico": (
        "Habla como un locutor de radio comercial: enérgico, rápido, entusiasta, "
        "proyectando la voz y marcando mucho las cifras."
    ),
    "confidencial": (
        "Habla en voz muy baja, casi susurrando, como contando algo confidencial "
        "a alguien que está sentado al lado. Íntimo y lento."
    ),
}

import wave


def duracion_segundos(ruta_wav: Path) -> float:
    """Duración de un WAV, usando solo la librería estándar.

    Ojo: la API devuelve un WAV *de streaming*, con los tamaños del header en
    0xFFFFFFFF porque al escribirlo todavía no sabe cuánto va a durar. Por eso
    `getnframes()` devuelve basura y hay que medir el archivo real.
    """
    with wave.open(str(ruta_wav)) as w:
        bytes_por_segundo = w.getframerate() * w.getnchannels() * w.getsampwidth()

    datos = ruta_wav.read_bytes()
    inicio_audio = datos.find(b"data") + 8  # 'data' + 4 bytes de tamaño
    return (len(datos) - inicio_audio) / bytes_por_segundo


duraciones: dict[str, float] = {}

if HAY_OPENAI:
    for nombre, instruccion in DIRECCIONES.items():
        # WAV en vez de MP3 para poder medir la duración exacta con `wave`
        ruta = hablar(FRASE, voz="marin", instrucciones=instruccion,
                      nombre_archivo=f"02_{nombre}.wav")
        duraciones[nombre] = duracion_segundos(ruta)
        print(f"── {nombre}" + ("  (sin instrucciones)" if instruccion is None else ""))
        display(Audio(filename=str(ruta)))
else:
    print("⚠️ Falta OPENAI_API_KEY — se salta esta sección.")

### El efecto, en números

La diferencia se oye, pero conviene también **medirla**: es exactamente el mismo texto y la misma voz, así que toda diferencia de duración viene de la actuación. Si las cuatro versiones duran distinto, el modelo cambió el **ritmo**, no solo el timbre.

In [ ]:
if duraciones:
    base = duraciones["neutra"]
    print(f"{'dirección':<20} {'duración':>9} {'vs. neutra':>12}")
    print("-" * 44)
    for nombre, segundos in duraciones.items():
        delta = "—" if nombre == "neutra" else f"{(segundos / base - 1) * 100:+.0f}%"
        print(f"{nombre:<20} {segundos:>8.2f}s {delta:>12}")
    dispersion = (max(duraciones.values()) / min(duraciones.values()) - 1) * 100
    print(f"\nMismo texto ({len(FRASE)} caracteres), misma voz (marin). "
          "Lo único que cambió fue la instrucción.")
    print(f"Dispersión entre la más larga y la más corta: {dispersion:.0f}%")

Acá hay que tener cuidado con la conclusión, y vale la pena detenerse porque es un buen ejemplo de cómo *no* medir un modelo generativo.

Corre la celda dos o tres veces y mira la dispersión. Preparando esta lección la corrimos varias veces y salió **entre 5% y 30%**, sin patrón: a veces la versión "confidencial" es la más larga, a veces la más corta, a veces todas quedan casi iguales. La síntesis no es determinista y "habla pausado" no se traduce en un factor de velocidad estable.

Entonces, ¿la instrucción hace algo? Sí — pero eso lo establece el **oído**, no este número. Escuchando los cuatro audios la diferencia de carácter es evidente e inmediata; la duración es solo un efecto secundario ruidoso de ese cambio, y con cuatro muestras de una frase no alcanza para medir nada.

La conclusión práctica es doble:

- `instructions` es un control de **estilo**, no de **timing**. Sirve perfectamente para fijar el carácter de una voz de marca. No sirve donde necesites duración exacta — un spot de 30 segundos, un doblaje que calce con video, un mensaje que entre en una ventana de IVR. Para eso: generar, medir, y reintentar o recortar el texto.
- Cuando evalúes un modelo generativo, **una corrida no es una medición**. Si un número te importa, córrelo varias veces y mira la dispersión antes de sacar conclusiones — es exactamente lo que hicimos en la lección 1 con el ruido, y por eso ahí el hallazgo era sólido y acá no.

### Por qué esto importa más de lo que parece

Tres consecuencias prácticas de poder escribir la dirección en vez de grabarla:

- **El tono se vuelve configuración, no producción.** Cambiar la personalidad de tu asistente telefónico es editar un string, no contratar de nuevo a un actor de voz ni reentrenar nada.
- **El tono puede ser dinámico.** Nada impide que la instrucción dependa del contexto: más pausado y empático si el cliente viene de una queja, más directo si es una confirmación de rutina.
- **Es un vector de riesgo nuevo.** Si la instrucción se arma concatenando texto de un usuario, alguien puede inyectar la actuación que quiera. Trátala como cualquier otro prompt: con las mismas precauciones.

> Un detalle de facturación: `gpt-4o-mini-tts` se cobra por **tokens**, no por caracteres — unos US$0,60 por millón de tokens de texto de entrada y US$12 por millón de tokens de audio de salida (≈ US$0,015 por minuto de audio). Las `instructions` también son tokens de entrada, pero son cortas y el costo dominante es el audio.

## Las 13 voces

`gpt-4o-mini-tts` trae 13 voces predefinidas. No son clonables ni personalizables — son estas y punto — pero cubren bastante rango. OpenAI recomienda **`marin`** y **`cedar`** como las de mejor calidad; son las más nuevas.

Escúchalas seguidas: la diferencia entre ellas es de timbre y registro, no de acento (todas hablan español con una fonética bastante neutra, sin acento regional marcado). Eso es justamente lo que ElevenLabs viene a resolver en la sección siguiente.

In [ ]:
VOCES = ["alloy", "ash", "ballad", "coral", "echo", "fable",
         "nova", "onyx", "sage", "shimmer", "verse", "marin", "cedar"]

if HAY_OPENAI:
    for voz in VOCES:
        ruta = hablar(f"Hola, soy {voz}, y esta es mi voz.", voz=voz,
                      nombre_archivo=f"03_voz_{voz}.mp3")
        print(f"── {voz}" + ("  ← recomendada" if voz in ("marin", "cedar") else ""))
        display(Audio(filename=str(ruta)))
else:
    print("⚠️ Falta OPENAI_API_KEY — se salta el catálogo de voces.")

## ElevenLabs: el especialista

OpenAI resuelve el caso general con 13 voces. **ElevenLabs** es la empresa que se dedica solo a esto, y su propuesta es distinta en tres cosas:

- **Catálogo enorme y con acento.** Miles de voces de la comunidad, filtrables por idioma, acento, edad y caso de uso. Para un producto chileno, poder elegir una voz que suene **chilena** —y no a español neutro de doblaje— es una diferencia que los usuarios notan de inmediato.
- **Clonación.** Con unos minutos de audio se crea una voz nueva; es lo que usan los estudios de doblaje y los creadores de audiolibros.
- **Modelos por caso de uso.** `eleven_v3` para máxima expresividad, `eleven_flash_v2_5` cuando la latencia manda (agentes conversacionales), `eleven_turbo_v2_5` en el medio.

Veamos primero qué tiene disponible la cuenta.

In [ ]:
eleven = None

if not HAY_ELEVEN:
    print("⚠️ Falta ELEVENLABS_API_KEY — se salta toda la sección de ElevenLabs.")
else:
    from elevenlabs.client import ElevenLabs

    eleven = ElevenLabs()  # lee ELEVENLABS_API_KEY del entorno

    suscripcion = eleven.user.subscription.get()
    print(f"Plan: {suscripcion.tier} · "
          f"caracteres usados {suscripcion.character_count:,} de {suscripcion.character_limit:,}\n")

    print("Modelos de texto a voz disponibles:")
    for modelo in eleven.models.list():
        if getattr(modelo, "can_do_text_to_speech", False):
            print(f"  {modelo.model_id:<28} {modelo.name}")

In [ ]:
if eleven is not None:
    # Las etiquetas de cada voz son lo que hace útil el catálogo: idioma, acento, caso de uso.
    print(f"{'voice_id':<24} {'nombre':<42} {'idioma':<7} acento")
    print("-" * 96)
    for voz in eleven.voices.get_all().voices[:12]:
        etiquetas = getattr(voz, "labels", {}) or {}
        print(f"{voz.voice_id:<24} {voz.name[:40]:<42} "
              f"{etiquetas.get('language', '?'):<7} {etiquetas.get('accent', '?')}")

### Sintetizar con `eleven_v3`

Dos cosas a saber antes de correr la celda, porque es donde tropieza todo el mundo:

- Las voces **premade** (las que vienen con cualquier cuenta) funcionan en el plan gratis.
- Las voces de la **biblioteca de la comunidad** —incluidas las que tienen acento chileno— requieren **plan pago**. Con una cuenta gratis la API responde `402 paid_plan_required`. La celda maneja ese caso y sigue.

El plan gratis da 10.000 caracteres al mes, suficiente para esta lección de sobra.

In [ ]:
VOZ_PREMADE = "JBFqnCBsd6RMkjVDRZzb"   # George — disponible en cualquier plan
VOZ_BIBLIOTECA = "yytxkT3pNVMWDHn3KXrY"  # voz con acento chileno — requiere plan pago

TEXTO_EL = "Hola, y bienvenidos al Diplomado de Inteligencia Artificial Generativa."


def hablar_eleven(texto: str, voice_id: str, nombre_archivo: str,
                  modelo: str = "eleven_v3") -> Path | None:
    """Sintetiza con ElevenLabs; devuelve None si la voz exige plan pago."""
    try:
        respuesta = eleven.text_to_speech.convert(
            text=texto, voice_id=voice_id, model_id=modelo, output_format="mp3_44100_128"
        )
        # convert() devuelve un generador perezoso: la llamada HTTP recién ocurre
        # al consumirlo, así que hay que materializarlo DENTRO del try.
        audio = b"".join(respuesta)
    except Exception as error:
        detalle = getattr(error, "body", None) or {}
        codigo = detalle.get("detail", {}).get("code") if isinstance(detalle, dict) else None
        if codigo == "paid_plan_required":
            print(f"   ⏭️  {voice_id}: requiere plan pago (voz de biblioteca). Se salta.")
            return None
        raise

    destino = SALIDAS / nombre_archivo
    destino.write_bytes(audio)
    return destino


if eleven is not None:
    print("── voz premade (eleven_v3)")
    ruta = hablar_eleven(TEXTO_EL, VOZ_PREMADE, "04_eleven_premade.mp3")
    if ruta:
        display(Audio(filename=str(ruta)))

    print("\n── voz de biblioteca con acento chileno (eleven_v3)")
    ruta = hablar_eleven(TEXTO_EL, VOZ_BIBLIOTECA, "05_eleven_chilena.mp3")
    if ruta:
        display(Audio(filename=str(ruta)))
    else:
        print("   (Con plan pago acá se escucharía la diferencia de acento, que es el punto)")

### Clonación de voz: la parte que hay que conversar

ElevenLabs clona una voz con pocos minutos de audio, y la calidad ya es suficiente para engañar a alguien que no esté prestando atención. No lo hacemos en esta lección — requiere plan pago y consentimiento de la persona — pero la conversación es parte del contenido de la clase:

- **Consentimiento explícito y verificable.** La voz de una persona es un dato biométrico. Clonar la voz de un ejecutivo para el IVR de la empresa necesita autorización escrita, igual que usar su imagen.
- **El fraude por voz clonada ya es un problema real**, no una hipótesis: llamadas suplantando a un familiar o a un gerente pidiendo transferencias urgentes. Si tu organización autentica a alguien *por su voz*, esa medida hoy está comprometida.
- **Trazabilidad.** Conviene marcar el audio sintético que produces (watermarking, metadatos) y guardar registro de qué se generó y con qué voz.

La capacidad técnica está resuelta; el problema pasó a ser de gobierno y de política interna.

## La decisión: costo y latencia

Para elegir hace falta medir. La latencia de la síntesis va a ser una de las tres que sumemos en el sándwich de la lección 3, así que conviene tener el número real y no la intuición.

Medimos el tiempo total hasta tener el archivo completo, con la misma frase en ambos proveedores.

In [ ]:
import time

FRASE_BENCH = (
    "Buenas tardes. Su cita quedó confirmada para el martes a las diez de la mañana."
)
REPETICIONES = 3


def cronometrar(funcion, veces: int = REPETICIONES) -> list[float]:
    tiempos = []
    for _ in range(veces):
        inicio = time.perf_counter()
        funcion()
        tiempos.append(time.perf_counter() - inicio)
    return tiempos


mediciones: dict[str, list[float]] = {}

if HAY_OPENAI:
    mediciones["OpenAI · gpt-4o-mini-tts"] = cronometrar(
        lambda: hablar(FRASE_BENCH, voz="marin", nombre_archivo="06_bench_openai.mp3")
    )

if eleven is not None:
    for modelo in ("eleven_v3", "eleven_flash_v2_5"):
        try:
            mediciones[f"ElevenLabs · {modelo}"] = cronometrar(
                lambda m=modelo: hablar_eleven(
                    FRASE_BENCH, VOZ_PREMADE, f"06_bench_{m}.mp3", modelo=m
                )
            )
        except Exception as error:
            print(f"{modelo}: no se pudo medir ({type(error).__name__})")

if mediciones:
    print(f"{'proveedor · modelo':<34} {'mediana':>9} {'mín':>8} {'máx':>8}")
    print("-" * 62)
    for nombre, tiempos in mediciones.items():
        ordenados = sorted(tiempos)
        mediana = ordenados[len(ordenados) // 2]
        print(f"{nombre:<34} {mediana:>8.2f}s {min(tiempos):>7.2f}s {max(tiempos):>7.2f}s")
    print(f"\n(frase de {len(FRASE_BENCH)} caracteres · {REPETICIONES} repeticiones · "
          "audio completo, no primer byte)")
else:
    print("⚠️ Sin llaves no hay nada que medir.")

### Cómo elegir

El resultado más útil de la medición no es cuál proveedor gana, sino **la brecha entre los modelos expresivos y los `flash`**: los dos modelos expresivos quedan en el mismo orden de magnitud (~2-2,5 s para esta frase), y `eleven_flash_v2_5` es unas **cinco veces más rápido**. Esa diferencia es la que decide si un agente conversacional se siente natural o se siente lento.

| | OpenAI `gpt-4o-mini-tts` | ElevenLabs |
|---|---|---|
| **Voces** | 13 fijas, sin acento regional | Miles, filtrables por idioma y **acento**; + clonación |
| **Control del tono** | `instructions` en lenguaje natural | Ajustes de estabilidad/similitud y etiquetas de audio |
| **Costo** | ≈ US$0,015 por minuto de audio | Por caracteres, según plan; free 10.000/mes |
| **Latencia (medida arriba)** | ~2,3 s | `eleven_v3` ~2,5 s · **`eleven_flash_v2_5` ~0,5 s** |
| **Plan gratis** | No (requiere cuenta con saldo) | Sí, 10.000 caracteres/mes |
| **Voces de biblioteca por API** | — | Requieren **plan pago** |
| **Fuerte en** | Ya tienes la llave de OpenAI y quieres dirigir el tono por texto | La voz *es* el producto: marca, acento local, personaje, doblaje |

Regla práctica: si estás construyendo un agente conversacional y la voz es un medio, parte con OpenAI — una llave menos que administrar y el control por `instructions` alcanza. Si la voz es el producto, o necesitas que suene de un país en particular, ElevenLabs vale la llave extra (con plan pago, porque las voces con acento son de biblioteca).

Y para cualquier cosa en tiempo real, elige el modelo por latencia antes que por expresividad. Guarda ese ~2,3 s: en la lección 3 lo vamos a sumar con la transcripción y con el agente, y ahí se va a entender por qué la lección 4 existe.

> Ojo con cómo leer estos números: son de **audio completo**, con pocas repeticiones y desde una conexión particular. En producción lo que importa es el **primer byte** (*time to first byte*), porque se puede empezar a reproducir mientras el resto se genera. Tómalos como orden de magnitud, no como benchmark.

## Qué nos llevamos

- La síntesis moderna se **dirige**: `instructions` convierte el tono en configuración editable, no en una sesión de grabación. Es el cambio más importante de esta generación de modelos de TTS — pero es control de **estilo**, no de **duración**: el efecto sobre el ritmo existe y se oye, y a la vez es tan inestable entre corridas que no se puede usar para calzar tiempos.
- `tts-1` y `tts-1-hd` son la generación anterior — menos voces y sin `instructions`. Si ves código con `tts-1`, está desactualizado.
- OpenAI cubre el caso general; **ElevenLabs** gana en catálogo, acento local y clonación, y trae la conversación de consentimiento y fraude que hay que tener con el área legal antes que con la técnica.
- Elige el modelo por **latencia** cuando la voz sea conversacional, no por expresividad.

Ya tenemos las dos mitades: la lección 1 convierte voz en texto y esta convierte texto en voz. En la **lección 3** las juntamos alrededor de un agente — el método **sándwich** — y medimos qué precio se paga por encadenar tres modelos.